# 중간발표 탐지 성능 보고서 (Colab)

모든 데이터셋을 최신 라벨 기준으로 새로 만들고, 표본 없이 전수 채점해서 `report/detection/`에 파일 5개를 만든다. 지연 측정은 하지 않는다(PC에서 따로 잰다).

- **채점 구성:** 시스템, 시스템 + v9, 시스템 + Qwen, CredSweeper+ML, 어댑터: credsweeper, 어댑터: gitleaks, 기존 3개 합집합, 기존 다수결. 채점기 점검으로 oracle과 identity도 돌린다.
- **ML 모듈:** "시스템 + v9"와 "시스템 + Qwen"은 시스템 기본 설정에 ML 모듈 하나를 플러그인 자체 스위치(`PG_DETECTOR_ENABLE`)로 켠 것이다. 모듈이 실제로 돌지 않으면 그 실행은 점수를 내지 않고 멈춘다. **GPU 런타임을 쓴다**(Qwen).
- **제외한 구성:** CredSweeper(ML off), gitleaks 단독, 어댑터: 자체 규칙은 이미 따로 분석했으므로 돌리지 않는다(① `ALREADY_DONE`). 그 결과가 필요한 칸은 —로 두고 이유를 적는다. 해당 칸은 README 문장 1, CredSweeper가 놓친 정답 표, 그 어댑터만 가린 정답 수다.
- **고치지 않는 것:** 시스템 코드와 채점기 코드. 어댑터 단독·합집합·다수결은 맨 위 헬퍼가 시스템의 `pgdetector` 클래스로 조립한다. 시스템 설정 파일은 바꾸지 않는다.
- **비밀값:** 보고서의 예시는 `⟦유형:길이자⟧`로 가린다. 보고서를 쓴 뒤 정답 값이 들어 있으면 그 파일을 지우고 멈춘다.
- **사람이 할 일:** ⑨의 분류 검수(무작위 30개를 보고 판정 입력).
- **끊겨도 이어서:** 만든 데이터와 끝난 채점은 Drive에 남는다. 런타임이 끊기면 ①부터 다시 실행하면 끝난 단계는 건너뛴다.

순서: ⓪ 헬퍼 → ① 설정 → ② 저장소·Drive → ③ 패키지 → ④ 원본 받기 → ⑤ 데이터 생성·검증 → ⑥ 시스템 동작 확인 → ⑦ 채점 → ⑧ 보고서 → ⑨ 분류 검수 → ⑩ 채팅 보고용 요약

## ⓪ 헬퍼

분석용 코드다. `/content/pg_report.py`로 저장해서 아래 셀들이 쓴다. 규칙을 고쳐야 하면 ⑨에서 고친다. 이 셀은 다시 실행하면 처음 상태로 돌아간다.

In [ ]:
%%writefile /content/pg_report.py
"""Midterm detection report helpers (analysis code only).

Runs the metric-detection scorer (evaluation.cli.execute_run) and the PromptGuard detector as they are: the
composed configurations (one adapter, the three-adapter union and majority) are built from the system's own
pgdetector classes in this process, never by editing the system or the scorer. Every file written under report/
masks secret values as ⟦TYPE:N자⟧.
"""
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path(os.environ.get("PG_ROOT", "/content"))
REPO = ROOT / "promptguard-data"
MD = REPO / "metric-detection"
SYSTEM_ROOT = ROOT / "promptguard-claude-demoV0"
WORK = Path(os.environ.get("PG_WORK", ROOT / "work"))
RUNS, CHUNKS, LOGS = WORK / "runs", WORK / "chunks", WORK / "logs"
STATE = WORK / "state.json"
REPORT = Path(os.environ.get("PG_REPORT", REPO / "report" / "detection"))
GITLEAKS_CACHE = WORK / "gitleaks-cache.sqlite"

SCORED = ("prompt", "instructions", "tool_output")
TYPES = ("PASSWORD", "TOKEN", "ACCESS_KEY", "PRIVATE_KEY", "SECRET")
THREE = ("credsweeper", "gitleaks", "promptguard-rules")
SYSTEM_ADAPTERS = ("credsweeper", "gitleaks", "promptguard-rules", "shell-aware")

# (key, label, stems or a glob over data_test)
DATASETS = [
    ("builtin", "내장 코퍼스", ["sessions"]),
    ("example", "템플릿 예시", ["sessions_from_example"]),
    ("v2", "에이전트 세션 v2", ["sessions_2"]),
    ("creddata", "CredData", ["sessions_from_CredData"]),
    ("nemotron", "Nemotron-PII", ["sessions_from_Nemotron-PII"]),
    ("noseyparker", "Nosey Parker", ["sessions_from_noseyparker"]),
    ("openhands", "openhands-feedback", ["sessions_from_openhands-feedback"]),
    ("swegym", "SWE-Gym", ["sessions_from_SWE-Gym"]),
    # Whole ML training builds, split ignored (ml_full_copies): the synthetic templates, and privesc's ten scenarios.
    ("ml_test", "ML 평가 test", "sessions_mlfull_test_*.jsonl"),
    ("ml_test_ood", "ML 평가 test_ood", "sessions_mlfull_ood_*.jsonl"),
]
LABEL = {key: label for key, label, _ in DATASETS}
SYNTH = ("builtin", "example", "v2")
ONLY = [k for k in os.environ.get("PG_ONLY", "").split(",") if k]  # limit to some dataset keys (tests, dry runs)
ORIGIN_KIND = {"authored": "직접 작성", "external": "외부 변환", "recorded": "실제 기록"}

# Run order is the priority the request gives; the tables use DISPLAY.
CONFIGS = [
    ("system", "시스템"), ("credsweeper", "CredSweeper"), ("gitleaks", "gitleaks"), ("union3", "기존 3개 합집합"),
    ("adapter_rules", "어댑터: 자체 규칙"), ("credsweeper_ml", "CredSweeper+ML"), ("majority3", "기존 다수결"),
    ("system_v9", "시스템 + v9"), ("adapter_credsweeper", "어댑터: credsweeper"), ("adapter_gitleaks", "어댑터: gitleaks"),
    ("system_qwen", "시스템 + Qwen"),
]
CHECKS = [("oracle", "oracle"), ("identity", "identity")]
NAME = dict(CONFIGS + CHECKS)
DISPLAY = ["system", "system_v9", "system_qwen", "credsweeper", "credsweeper_ml", "gitleaks", "adapter_credsweeper", "adapter_gitleaks",
           "adapter_rules", "union3", "majority3"]
COMPOSE = {"adapter_credsweeper": (["credsweeper"], "any"), "adapter_gitleaks": (["gitleaks"], "any"),
           "adapter_rules": (["promptguard-rules"], "any"), "union3": (list(THREE), "any"),
           "majority3": (list(THREE), "majority")}
# The system with one of its ML modules switched on through its own PG_DETECTOR_ENABLE (detector.json has them off).
ML = {"system_v9": "ml-v9", "system_qwen": "ml-qwen"}
GPU_CONFIGS = {"system_qwen"}  # one process at a time: each loads Qwen on the GPU
PG_CONFIGS = {"system", *ML, *COMPOSE}
# Configurations analysed elsewhere: not run here; their cells stay — with this reason. Set from the notebook.
EXCLUDED: dict[str, str] = {}


# ---------------------------------------------------------------- small utilities

def sh(cmd, cwd=None, log=None, check=True, env=None):
    print("$", cmd, flush=True)
    if log:
        Path(log).parent.mkdir(parents=True, exist_ok=True)
        with open(log, "a", encoding="utf-8") as handle:
            code = subprocess.run(cmd, shell=True, cwd=cwd, stdout=handle, stderr=subprocess.STDOUT, env=env).returncode
    else:
        code = subprocess.run(cmd, shell=True, cwd=cwd, env=env).returncode
    if check and code:
        raise RuntimeError(f"exit {code}: {cmd}" + (f" (log: {log})" if log else ""))
    return code


def clean_env():
    return {key: value for key, value in os.environ.items() if not key.startswith("PG_")}


def read_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def load_state():
    return json.loads(STATE.read_text()) if STATE.is_file() else {"stages": {}, "notes": []}


def save_state(state):
    STATE.parent.mkdir(parents=True, exist_ok=True)
    STATE.write_text(json.dumps(state, ensure_ascii=False, indent=2))


class stage:
    """with stage("데이터 생성"): ... records wall time per step in state.json (shown in README)."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.started = time.time()

    def __exit__(self, kind, *_):
        state = load_state()
        entry = state["stages"].setdefault(self.name, {"seconds": 0.0})
        entry["seconds"] += time.time() - self.started
        entry["ok"] = kind is None
        save_state(state)
        print(f"[{self.name}] {'done' if kind is None else 'FAILED'} in {time.time() - self.started:.0f}s")


def datasets():
    """key → [(stem, sessions, gold)] for the built evaluation files; stray files in data_test are reported."""
    found, listed = {}, set()
    for key, _label, stems in DATASETS:
        if ONLY and key not in ONLY:
            continue
        paths = sorted((MD / "data_test").glob(stems)) if isinstance(stems, str) else \
            [MD / "data_test" / f"{stem}.jsonl" for stem in stems]
        found[key] = []
        for path in paths:
            listed.add(path.name)
            gold = MD / "data_answer" / ("gold" + path.name[len("sessions"):])
            if path.is_file() and gold.is_file():
                found[key].append((path.stem, path, gold))
    stray = [] if ONLY else sorted(path.name for path in (MD / "data_test").glob("sessions*.jsonl") if path.name not in listed)
    return found, stray


# ---------------------------------------------------------------- chunks and runs

def make_chunks(target_items=1500):
    """Split every dataset into whole-session chunks of about target_items scored items (sessions and gold)."""
    found, stray = datasets()
    if stray:
        print("warning: not evaluated (not in the dataset list):", ", ".join(stray))
    chunks = []
    for key, files in found.items():
        for stem, sessions, gold in files:
            folder = CHUNKS / stem
            stamp = folder / "source.sha256"
            source = f"{sha256(sessions)} {sha256(gold)} {target_items}"
            if not (stamp.is_file() and stamp.read_text() == source):
                shutil.rmtree(folder, ignore_errors=True)
                folder.mkdir(parents=True)
                gold_rows = defaultdict(list)
                for line in open(gold, encoding="utf-8"):
                    if line.strip():
                        gold_rows[json.loads(line)["session_id"]].append(line if line.endswith("\n") else line + "\n")
                parts, current, weight = [], [], 0
                for line in open(sessions, encoding="utf-8"):
                    if not line.strip():
                        continue
                    record = json.loads(line)
                    current.append((record["session_id"], line if line.endswith("\n") else line + "\n"))
                    weight += sum(item["channel"] in SCORED for item in record["items"])
                    if weight >= target_items:
                        parts.append((current, weight))
                        current, weight = [], 0
                if current:
                    parts.append((current, weight))
                for index, (rows, weight) in enumerate(parts):
                    name = f"{index:03d}"
                    (folder / f"sessions_{name}.jsonl").write_text("".join(line for _sid, line in rows), encoding="utf-8")
                    (folder / f"gold_{name}.jsonl").write_text(
                        "".join(g for sid, _line in rows for g in gold_rows.get(sid, [])), encoding="utf-8")
                    (folder / f"weight_{name}").write_text(str(weight))
                stamp.write_text(source)
            for weight_file in sorted(folder.glob("weight_*")):
                name = weight_file.name[len("weight_"):]
                chunks.append({"dataset": key, "stem": stem, "name": name, "weight": int(weight_file.read_text()),
                               "sessions": str(folder / f"sessions_{name}.jsonl"),
                               "gold": str(folder / f"gold_{name}.jsonl")})
    return chunks


def run_dir(config, chunk):
    return RUNS / config / chunk["stem"] / chunk["name"]


def finished(config, chunk):
    return next(iter(sorted(run_dir(config, chunk).glob("*/metrics.json"))), None)


RUN_CHUNK = r'''
"""One scorer run over one chunk: python run_chunk.py CONFIG SESSIONS GOLD OUT SYSTEM_ROOT REPO GITLEAKS_CACHE"""
import subprocess, sys
from pathlib import Path
from types import SimpleNamespace

COMPOSE = __COMPOSE__
ML = __ML__
PG = {"system", *ML, *COMPOSE}


def factory():
    if config in PG:
        from evaluation.adapters.promptguard_adapter import PromptGuardAdapter
        worker = load_worker(system_root)
        if config == "system":
            return PromptGuardAdapter(worker)
        if config in ML:  # PG_DETECTOR_ENABLE was set before load_detector() runs; a skipped module stops the run
            return PromptGuardAdapter(worker, required_adapters=[ML[config]])
        names, vote = COMPOSE[config]
        base = worker.load_detector()
        skipped = {entry["name"]: entry["reason"] for entry in base.skipped}
        missing = [name for name in names if name not in base.names]
        if missing:
            raise ValueError("adapter(s) not running: " + ", ".join(f"{n} ({skipped.get(n, 'not configured')})" for n in missing))
        kept = [adapter for adapter in base.adapters if adapter.name in names]
        dropped = [{"name": name, "reason": "not selected"} for name in base.names if name not in names]
        # Plain vote among the kept adapters: no candidate-only adapters, the pre-union candidate policy.
        detector = worker.Detector(kept, worker.ConsensusPolicy.parse(vote), base.skipped + dropped, (), "majority")
        return PromptGuardAdapter(SimpleNamespace(load_detector=lambda: detector))
    if config in ("credsweeper", "credsweeper_ml"):
        from evaluation.adapters.credsweeper_adapter import CredSweeperAdapter
        return CredSweeperAdapter(ml=config == "credsweeper_ml")
    if config == "gitleaks":
        from evaluation.adapters.gitleaks_adapter import GitleaksAdapter
        findings = Path(out) / "gitleaks_findings.jsonl"
        if not findings.is_file():
            subprocess.run([sys.executable, str(Path(repo) / "metric-detection/scripts/run_gitleaks.py"),
                            "--sessions", sessions, "--out", str(findings)], check=True)
        return GitleaksAdapter(findings, sessions)
    from evaluation.adapters.reference import IdentityAdapter, OracleAdapter
    return OracleAdapter() if config == "oracle" else IdentityAdapter()

if __name__ == "__main__":  # CredSweeper's multiprocessing children re-import this file
    config, sessions, gold, out, system_root, repo, cache_path = sys.argv[1:8]
    sys.path.insert(0, str(Path(repo) / "metric-detection"))
    sys.path.insert(0, str(Path(repo) / "dashboard"))
    import os
    for key in [k for k in os.environ if k.startswith("PG_")]:
        del os.environ[key]
    if config in ML:
        os.environ["PG_DETECTOR_ENABLE"] = ML[config]
    cache = None
    if config in PG:
        from pipeline.gitleaks_cache import install  # caches `gitleaks stdin` runs at the process boundary
        cache = install(Path(cache_path))
    from evaluation.cli import execute_run
    from evaluation.adapters.system_root import load_worker
    Path(out).mkdir(parents=True, exist_ok=True)
    try:
        code = execute_run(sessions, gold, factory, label=config,
                           use="ml_eval" if config == "credsweeper_ml" or config in ML else "rule_eval",
                           out_root=out, system_root=Path(system_root) if config in PG else None)
    finally:
        if cache is not None:
            cache.close()
    sys.exit(code)
'''

SCHEDULER = r'''
"""Background scheduler: python scheduler.py TASKS.json PROGRESS.json JOBS [MIRROR]. Logs per task."""
import json, os, shutil, subprocess, sys, time
from pathlib import Path

tasks = json.load(open(sys.argv[1])); progress_path = Path(sys.argv[2]); jobs = int(sys.argv[3])
mirror = Path(sys.argv[4]) if len(sys.argv) > 4 and sys.argv[4] else None
env = {k: v for k, v in os.environ.items() if not k.startswith("PG_")}
def done(task): return any(Path(task["out"]).glob("*/metrics.json"))
previous = json.loads(progress_path.read_text()).get("seconds", {}) if progress_path.is_file() else {}
state = {"started_at": time.time(), "total": len(tasks), "done": 0, "skipped": 0, "failed": [], "running": [],
         "seconds": previous, "pid": os.getpid(), "finished_at": None}  # seconds accumulate over restarts
def save():
    tmp = progress_path.with_suffix(".tmp"); tmp.write_text(json.dumps(state, ensure_ascii=False)); os.replace(tmp, progress_path)
pending = []
for task in tasks:
    if done(task): state["skipped"] += 1; state["done"] += 1
    else: pending.append(task)
running = []
names = {t["name"] for t in pending}
def ready(task):  # "after": a task whose gitleaks runs fill the shared cache first
    after = task.get("after")
    return not after or after not in names or after not in {t["name"] for t in pending} | {t["name"] for t, *_ in running}
while pending or running:
    while len(running) < jobs:
        gpu_busy = sum(1 for t, *_ in running if t.get("gpu"))
        task = next((t for t in pending if ready(t) and not (t.get("gpu") and gpu_busy >= 1)), None)
        if task is None:
            break
        pending.remove(task)
        shutil.rmtree(task["out"], ignore_errors=True); Path(task["log"]).parent.mkdir(parents=True, exist_ok=True)
        log = open(task["log"], "w")
        process = subprocess.Popen(task["argv"], stdout=log, stderr=subprocess.STDOUT, env=env)
        running.append((task, process, log, time.time()))
    state["running"] = [t["name"] for t, *_ in running]; save()
    time.sleep(2)
    for entry in list(running):
        task, process, log, started = entry
        if process.poll() is None: continue
        log.close(); running.remove(entry)
        state["seconds"][task["config"]] = state["seconds"].get(task["config"], 0) + time.time() - started
        if process.returncode == 0 and done(task):
            state["done"] += 1
            if mirror:
                target = mirror / Path(task["out"]).relative_to(Path(task["runs"]))
                shutil.rmtree(target, ignore_errors=True); shutil.copytree(task["out"], target)
        else:
            lines = Path(task["log"]).read_text(errors="replace").strip().splitlines()
            state["failed"].append({"name": task["name"], "exit": process.returncode, "last": lines[-1][:300] if lines else ""})
state["running"] = []; state["finished_at"] = time.time(); save()
'''


def start(configs=None, jobs=None, mirror=None, chunks=None):
    """Start the background scheduler for configs × chunks (priority order). Returns its pid."""
    configs = [c for c in (configs or [name for name, _ in CONFIGS] + [name for name, _ in CHECKS]) if c not in EXCLUDED]
    chunks = chunks or make_chunks()
    WORK.mkdir(parents=True, exist_ok=True)
    script = WORK / "run_chunk.py"
    script.write_text(RUN_CHUNK.replace("__COMPOSE__", repr(COMPOSE)).replace("__ML__", repr(ML)))
    (WORK / "scheduler.py").write_text(SCHEDULER)
    tasks = []
    for config in configs:
        for chunk in sorted(chunks, key=lambda c: -c["weight"]):
            out = run_dir(config, chunk)
            name = f"{config}/{chunk['stem']}/{chunk['name']}"
            after = (f"system/{chunk['stem']}/{chunk['name']}"
                     if config in ("union3", "majority3", "adapter_gitleaks", *ML) and "system" in configs else None)
            tasks.append({"name": name, "config": config, "out": str(out), "runs": str(RUNS), "after": after,
                          "gpu": config in GPU_CONFIGS,
                          "log": str(LOGS / config / f"{chunk['stem']}_{chunk['name']}.log"),
                          "argv": [sys.executable, str(script), config, chunk["sessions"], chunk["gold"], str(out),
                                   str(SYSTEM_ROOT), str(REPO), str(GITLEAKS_CACHE)]})
    (WORK / "tasks.json").write_text(json.dumps(tasks))
    if mirror:  # resume: bring back runs finished in an earlier Colab session
        mirror = Path(mirror)
        for metrics in mirror.glob("*/*/*/*/metrics.json"):
            target = RUNS / metrics.parent.parent.relative_to(mirror)
            if not any(target.glob("*/metrics.json")):
                shutil.copytree(metrics.parent.parent, target, dirs_exist_ok=True)
    progress = WORK / "progress.json"  # mark unfinished before the scheduler starts, keeping accumulated seconds
    previous = json.loads(progress.read_text()) if progress.is_file() else {}
    progress.write_text(json.dumps({**previous, "finished_at": None, "started_at": time.time(), "done": 0,
                                    "total": len(tasks), "skipped": 0, "failed": [], "running": []}))
    process = subprocess.Popen([sys.executable, str(WORK / "scheduler.py"), str(WORK / "tasks.json"),
                                str(WORK / "progress.json"), str(jobs or os.cpu_count() or 2), str(mirror or "")],
                               stdout=open(WORK / "scheduler.log", "w"), stderr=subprocess.STDOUT,
                               env=clean_env(), start_new_session=True)
    print(f"scheduler pid {process.pid}: {len(tasks)} tasks, {jobs or os.cpu_count()} at a time")
    return process.pid


def status():
    path = WORK / "progress.json"
    if not path.is_file():
        print("no scheduler progress yet")
        return None
    state = json.loads(path.read_text())
    elapsed = (state["finished_at"] or time.time()) - state["started_at"]
    print(f"{state['done']}/{state['total']} done ({state['skipped']} reused), {len(state['failed'])} failed, "
          f"{elapsed / 60:.1f} min" + (" — finished" if state["finished_at"] else f", running: {', '.join(state['running'])}"))
    for failure in state["failed"]:
        print("  FAILED", failure["name"], "exit", failure["exit"], "→", failure["last"])
    return state


# ---------------------------------------------------------------- system check

def check_system(sample_per_dataset=40, port=18799):
    """The "시스템" row must be the real system: its own config (union candidate policy, every configured adapter,
    gitleaks running), the worker's HTTP answers equal to the in-process scan the scorer uses, and the mod's
    substitution equal to the scorer's (the scorer's own contract test runs register.mjs under Node)."""
    import urllib.request
    sys.path.insert(0, str(MD))
    from evaluation.adapters.system_root import load_worker
    worker = load_worker(SYSTEM_ROOT)
    detector = worker.load_detector()
    described = detector.describe()
    problems = []
    names = [a["name"] for a in described["adapters"]]
    if described["candidate_policy"] != "union" or described["required_votes"] != 1:
        problems.append(f"candidate_policy={described['candidate_policy']}, required_votes={described['required_votes']}")
    if set(names) != set(SYSTEM_ADAPTERS):
        problems.append(f"adapters {names} != {list(SYSTEM_ADAPTERS)}")
    if described["skipped"]:
        problems.append(f"skipped {described['skipped']}")
    config = json.loads((SYSTEM_ROOT / "src/detector/detector.json").read_text())
    enabled = [e["name"] for e in config["adapters"] if e.get("enabled", True)]
    if set(enabled) != set(names):
        problems.append(f"detector.json enables {enabled}, running {names}")

    found, _ = datasets()
    rng = random.Random(20261029)
    texts = []
    for files in found.values():
        pool = [(item["text"], {"prompt": "user_prompt", "instructions": "claude_md", "tool_output": "tool_result"}[item["channel"]])
                for _stem, sessions, _gold in files for record in read_jsonl(sessions) for item in record["items"]
                if item["channel"] in SCORED]
        texts += rng.sample(pool, min(sample_per_dataset, len(pool)))
    env = {**clean_env(), "PG_DETECTOR_PORT": str(port), "PG_DETECTOR_MODE": "normal"}
    process = subprocess.Popen([sys.executable, str(SYSTEM_ROOT / "src/detector/worker.py")], cwd=SYSTEM_ROOT, env=env,
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    mismatched = 0
    try:
        for _ in range(240):
            try:
                urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2)
                break
            except OSError:
                time.sleep(0.5)
        else:
            raise RuntimeError("worker did not start")
        keep = ("start", "end", "type", "offset_unit", "rule_id")  # value_hash is an HMAC with a per-process key
        for index in range(0, len(texts), 64):
            batch = texts[index:index + 64]
            body = json.dumps({"request_id": "check", "items": [{"id": i, "text": t, "source": s}
                                                                 for i, (t, s) in enumerate(batch)]}).encode()
            request = urllib.request.Request(f"http://127.0.0.1:{port}/scan-batch", body,
                                             {"content-type": "application/json"})
            answer = json.loads(urllib.request.urlopen(request, timeout=300).read())
            for (text, source), item in zip(batch, answer["items"]):
                local = detector.scan(text, source)
                pick = lambda result: (result["ok"], [tuple(f.get(k) for k in keep) for f in result["findings"]])
                mismatched += pick(local) != pick(item)
    finally:
        process.terminate()
        process.wait(timeout=30)
    if mismatched:
        problems.append(f"worker HTTP and in-process findings differ on {mismatched}/{len(texts)} texts")
    if not texts:
        problems.append("no texts to compare")
    contract = subprocess.run([sys.executable, "-m", "unittest", "evaluation.tests.test_system_contract"],
                              cwd=MD, capture_output=True, text=True, env={**clean_env(), "PROMPTGUARD_CLAUDE_ROOT": str(SYSTEM_ROOT)})
    tail = (contract.stderr.strip().splitlines() or ["?"])[-1]
    if contract.returncode != 0:
        problems.append(f"scorer/mod contract test failed: {tail}")
    result = {"ok": not problems, "problems": problems, "described": described, "texts_compared": len(texts),
              "contract_test": tail, "node": shutil.which("node") is not None}
    state = load_state()
    state["system_check"] = result
    save_state(state)
    print(json.dumps({k: v for k, v in result.items() if k != "described"}, ensure_ascii=False, indent=2))
    return result


# ---------------------------------------------------------------- loading results

EXPECTED = {"system": set(SYSTEM_ADAPTERS), **{c: set(SYSTEM_ADAPTERS) | {m} for c, m in ML.items()},
            **{name: set(names) for name, (names, _vote) in COMPOSE.items()}}


def load_results(chunks=None):
    """config → dataset → {"counts", "lines", "spans": [...], "problems": [...]} summed over its chunks."""
    chunks = chunks or make_chunks()
    results = {}
    for config, _ in CONFIGS + CHECKS:
        if config in EXCLUDED:
            continue
        per = {}
        for key in LABEL:
            mine = [c for c in chunks if c["dataset"] == key]
            if not mine:
                continue
            entry = {"counts": Counter(), "lines": 0, "spans": [], "problems": [], "blocked": 0, "failed_open": 0,
                     "describe": None, "missing": 0}
            for chunk in mine:
                metrics_path = finished(config, chunk)
                if metrics_path is None:
                    entry["missing"] += 1
                    continue
                run = metrics_path.parent
                metrics = json.loads(metrics_path.read_text())
                system = json.loads((run / "run_meta.json").read_text()).get("system", {})
                if config in EXPECTED:
                    detector = system.get("detector", {})
                    names = {a["name"] for a in detector.get("adapters", [])}
                    if names != EXPECTED[config]:
                        entry["problems"].append(f"{chunk['stem']}/{chunk['name']}: adapters {sorted(names)}")
                    entry["describe"] = detector
                    entry["blocked"] += system.get("blocked_items", 0)
                    entry["failed_open"] += system.get("failed_open_items", 0)
                entry["counts"].update({k: v for k, v in metrics["counts"].items() if isinstance(v, int)})
                entry["lines"] += metrics["lines_total"]
                for row in read_jsonl(run / "per_span.jsonl"):
                    if row["scored"]:
                        entry["spans"].append({**row, "stem": chunk["stem"]})
            per[key] = entry
        results[config] = per
    return results


def usable(entry):
    return entry is not None and not entry["missing"] and not entry["problems"]


def scores(counts, lines):
    gold, edits = counts["gold_total"], counts["edits_total"]
    recall = counts["exact"] / gold if gold else None
    precision = counts["edits_tp"] / edits if edits else None
    def f(beta):
        if recall is None or precision is None:
            return None
        if recall == 0 and precision == 0:
            return 0.0
        b2 = beta * beta
        return (1 + b2) * precision * recall / (b2 * precision + recall)
    return {"recall": recall, "precision": precision, "f1": f(1), "f2": f(2),
            "fp_per_1k": counts["edits_fp"] / lines * 1000 if lines else None, "gold": gold,
            **{status: counts[status] / gold if gold else None for status in ("exact", "over", "partial", "missed")}}


def combined(results, config, keys):
    """Micro over the given datasets; None when any of them is missing or invalid."""
    entries = [results.get(config, {}).get(key) for key in keys if key in results.get(config, {})]
    if not entries or not all(usable(e) for e in entries):
        return None
    counts, lines = Counter(), 0
    for entry in entries:
        counts.update(entry["counts"])
        lines += entry["lines"]
    return scores(counts, lines)


# ---------------------------------------------------------------- miss groups

KEYWORD = re.compile(r"pass|pwd|secret|token|key|auth|cred|api|bearer|session|cookie|private|비번|비밀번호|암호", re.I)
GROUPS = [("D-1", "경계: 일부만 가림"), ("D-2", "경계: 넓게 가림"), ("E", "형식 깨짐·개인키"), ("B-1", "URL 속 비밀번호"),
          ("B-2", "URL 쿼리·서명"), ("B-3", "HTTP 헤더·쿠키"), ("A-2", "셸 명령 인자"), ("C", "자리는 맞는데 값이 걸러짐"),
          ("B-4", "규칙에 없는 키 이름"), ("A-1", "자연어 문장"), ("A-3", "함수 인자·배열 원소"), ("A-4", "값만 있는 줄"),
          ("기타", "기타")]
GROUP_NAME = dict(GROUPS)
BIG = [("A", "키 이름 없는 자리"), ("B", "규칙이 모르는 자리"), ("C", "값이 걸러짐"), ("D", "경계 틀림"), ("E", "형식 깨짐"),
       ("기타", "기타")]
RULE_CHANGES = []
COLUMN_NOTE = ("- 같은 정답도 두 구성에서 판정이 다르면 그룹이 바뀐다(예: 놓쳤던 정답을 넓게 가리면 D-2). "
               "그래서 '놓침 − 메움 + 새로 놓침'이 오른쪽 열과 그룹마다 같지는 않다.")  # what the 30-sample review changed in the rules below; written to miss_groups.md

RX = {
    "url_user": re.compile(r"://[^\s/:@]+:$"),
    "query": re.compile(r"[?&;][^\s=?&;]+=$"),
    "header": re.compile(r"(?i)(authorization\s*:\s*((bearer|basic|token)\s+)?|cookie\s*:\s*([^\s=;]+=)?|"
                         r"\bbearer\s+|\bbasic\s+|\bx-[a-z0-9-]+\s*:\s*)[\"']?$"),
    "flag": re.compile(r"(-p|-p'|--password[= ])$"),
    "shell": re.compile(r"sshpass|\becho\b.*\||printf|<<<|sudo -S|\bsend(line)?\b|--requirepass|os\.write"),
    "assign": re.compile(r"[\"']?([A-Za-z_가-힣][\w.\-가-힣]*)[\"']?\s*(=>|:=|=|:)\s*[\"'`(\[{]?\s*$"),
    "word": re.compile(r"(?i)\b(is|are|use|with|password|token|key)\b.{0,20}$"),
    "korean": re.compile(r"(은|는|이|:)\s*$"),
    "arg": re.compile(r"[,(]\s*[\"'`]?$"),
}


def context(text, start, width=100):
    line_start = text.rfind("\n", 0, start) + 1
    return text[max(line_start, start - width):start]


def classify(status, channel, text, start, end):
    value, ctx = text[start:end], context(text, start)
    if status == "partial":
        return "D-1"
    if status == "over":
        return "D-2"
    if value.startswith("-----BEGIN") or "PRIVATE KEY" in text[max(0, start - 60):start]:
        return "E"
    if RX["url_user"].search(ctx):
        return "B-1"
    if RX["query"].search(ctx):
        return "B-2"
    if RX["header"].search(ctx):
        return "B-3"
    if RX["flag"].search(ctx) or RX["shell"].search(ctx):
        return "A-2"
    match = RX["assign"].search(ctx)
    if match:
        return "C" if KEYWORD.search(match.group(1)) else "B-4"
    if channel == "prompt" or RX["word"].search(ctx) or RX["korean"].search(ctx):
        return "A-1"
    if RX["arg"].search(ctx):
        return "A-3"
    if not ctx.strip():
        return "A-4"
    return "기타"


def big(group):
    return group[0] if group[0] in "ABCDE" else "기타"


class Texts:
    """(stem, session, item) → (channel, text) and the item's gold spans, read once per dataset file."""

    def __init__(self):
        self.items, self.gold = {}, defaultdict(list)
        found, _ = datasets()
        for files in found.values():
            for stem, sessions, gold in files:
                for record in read_jsonl(sessions):
                    for item in record["items"]:
                        self.items[stem, record["session_id"], item["item_id"]] = (item["channel"], item["text"])
                for row in read_jsonl(gold):
                    self.gold[stem, row["session_id"], row["item_id"]].append(row["span"])


TOKENISH = re.compile(r"[A-Za-z0-9+/=_\-.]{12,}")


def scrub(fragment):
    """Mask token-like runs outside gold too, so a window never shows secret-looking material such as the rest of
    an over-masked token: 12+ characters with 3+ digits, or mixed case with a digit and no underscore. Identifiers
    such as AUTH0_CLIENT_SECRET stay readable (the key name decides the group)."""
    def mask(match):
        run = match.group()
        digits = sum(c.isdigit() for c in run)
        mixed = any(c.islower() for c in run) and any(c.isupper() for c in run) and digits and "_" not in run
        return f"⟦문자열:{len(run)}자⟧" if digits >= 3 or mixed else run
    return TOKENISH.sub(mask, fragment)


def glabel(g):
    return g if g == GROUP_NAME[g] else f"{g} {GROUP_NAME[g]}"


def masked(texts, stem, sid, iid, start, end):
    """⟦TYPE:N자⟧ for every gold span in a window of 50 characters before and 15 after; never the raw value."""
    channel, text = texts.items[stem, sid, iid]
    lo = max(text.rfind("\n", 0, start) + 1, start - 50)
    newline = text.find("\n", end)
    hi = min(end + 15, newline if newline != -1 else len(text))
    spans = sorted((s for s in texts.gold[stem, sid, iid] if s["start"] < hi and s["end"] > lo), key=lambda s: s["start"])
    out, cursor = [], lo
    for span in spans:
        a, b = max(span["start"], lo), min(span["end"], hi)
        if a < cursor:
            continue
        out.append(scrub(text[cursor:a]))
        out.append(f"⟦{span['type']}:{span['end'] - span['start']}자⟧")
        cursor = b
    out.append(scrub(text[cursor:hi]))
    shown = "".join(out).replace("\t", "⇥")
    if (start, end) not in {(s["start"], s["end"]) for s in spans}:  # the span itself is always masked
        shown = shown.replace(text[start:end], "⟦?⟧")
    return f"[{channel}] …{shown}…"


def entropy(value):
    counts = Counter(value)
    return -sum(c / len(value) * math.log2(c / len(value)) for c in counts.values()) if value else 0.0


def misses(results, config, texts):
    """Non-exact scored gold of a config with its group: list of dicts (no values)."""
    rows = []
    for key, entry in results.get(config, {}).items():
        if not usable(entry):
            continue
        for span in entry["spans"]:
            if span["status"] == "exact":
                continue
            channel, text = texts.items[span["stem"], span["session_id"], span["item_id"]]
            rows.append({"dataset": key, "stem": span["stem"], "session_id": span["session_id"], "item_id": span["item_id"],
                         "start": span["start"], "end": span["end"], "type": span["type"], "channel": channel,
                         "status": span["status"],
                         "group": classify(span["status"], channel, text, span["start"], span["end"])})
    return rows


def span_key(row):
    return row["stem"], row["session_id"], row["item_id"], row["start"], row["end"]


def review_sample(rows, texts, n=30, seed=20261029):
    rng = random.Random(seed)
    picked = rng.sample(rows, min(n, len(rows)))
    return [{"id": index + 1, "dataset": LABEL[row["dataset"]], "group": row["group"], "type": row["type"],
             "status": row["status"],
             "context": masked(texts, row["stem"], row["session_id"], row["item_id"], row["start"], row["end"])}
            for index, row in enumerate(picked)]


# ---------------------------------------------------------------- report writing

def fmt(value, digits=3):
    return "—" if value is None else f"{value:.{digits}f}"


def cell(results, config, key, metric, n_of, digits=3):
    value = combined(results, config, [key])
    text = fmt(value[metric] if value else None, digits)
    n = n_of.get(key)
    return f"{text} (n={n})" if value and n is not None and n < 30 else text


def table(header, rows):
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in row) + " |" for row in rows]
    return "\n".join(lines)


def dataset_stats():
    found, _ = datasets()
    stats = {}
    for key, files in found.items():
        if not files:
            continue
        s = {"origins": Counter(), "sessions": 0, "items": 0, "scored": 0, "unscored": 0,
             "types": Counter(), "channels": Counter(), "files": []}
        for stem, sessions, gold in files:
            channel_of = {}
            for record in read_jsonl(sessions):
                s["sessions"] += 1
                s["origins"][record.get("meta", {}).get("origin")] += 1
                for item in record["items"]:
                    s["items"] += 1
                    channel_of[record["session_id"], item["item_id"]] = item["channel"]
            for row in read_jsonl(gold):
                channel = channel_of[row["session_id"], row["item_id"]]
                if channel in SCORED:
                    s["scored"] += 1
                    s["types"][row["span"]["type"]] += 1
                    s["channels"][channel] += 1
                else:
                    s["unscored"] += 1
            s["files"].append({"file": sessions.name, "sessions_sha256": sha256(sessions)[:12], "gold_sha256": sha256(gold)[:12]})
        stats[key] = s
    return stats


def dist(counter, order):
    total = sum(counter.values())
    return ", ".join(f"{k} {counter[k]}" for k in order if counter.get(k)) if total else "—"


def environment():
    def run(cmd):
        try:
            return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout.strip()
        except Exception:  # noqa: BLE001
            return ""
    cpu = run("lscpu | sed -n 's/^Model name:[ \\t]*//p'") or platform.processor()
    memory = run("awk '/MemTotal/ {printf \"%.1f GB\", $2/1048576}' /proc/meminfo")
    wsl = "WSL" if "microsoft" in run("cat /proc/version").lower() else "WSL 아님"
    colab = "Google Colab" if "COLAB_RELEASE_TAG" in os.environ or Path("/content").is_dir() else platform.node()
    try:
        from importlib.metadata import version
        credsweeper = version("credsweeper")
    except Exception:  # noqa: BLE001
        credsweeper = "—"
    try:
        import torch
        torch_version = torch.__version__
        gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음"
    except Exception:  # noqa: BLE001
        torch_version, gpu = "—", "—"
    model = ROOT / "promptguard-model"
    return {"model_commit": run(f"git -C {model} log -1 --format='%H (%cs) %s'") if model.is_dir() else "",
            "torch": torch_version, "gpu": gpu,
            "machine": f"{colab}, {cpu}, {os.cpu_count()} CPU, 메모리 {memory}, {wsl}",
            "system_commit": run(f"git -C {SYSTEM_ROOT} log -1 --format='%H (%cs) %s'"),
            "data_commit": run(f"git -C {REPO} log -1 --format='%H (%cs) %s'"),
            "credsweeper": credsweeper, "gitleaks": run("gitleaks version") or "—", "python": sys.version.split()[0]}


def write_report(review=None):
    """Write the five report files from finished runs. review: {id: ("Y"|"N", correct group or "", memo)}."""
    REPORT.mkdir(parents=True, exist_ok=True)
    chunks = make_chunks()
    results = load_results(chunks)
    texts = Texts()
    stats = dataset_stats()
    state = load_state()
    keys = [key for key in LABEL if key in stats]
    n_of = {key: stats[key]["scored"] for key in keys}
    notes = defaultdict(list)  # file → footnotes

    progress_file = WORK / "progress.json"
    failed = defaultdict(list)
    for failure in (json.loads(progress_file.read_text()).get("failed", []) if progress_file.is_file() else []):
        failed[failure["name"].split("/")[0]].append(failure["last"])

    def problems_of(config):
        if config in EXCLUDED:
            return [EXCLUDED[config]]
        out = [f"실패: {line}" for line in dict.fromkeys(failed.get(config, []))][:2]
        for key, entry in results.get(config, {}).items():
            if entry["missing"]:
                out.append(f"{LABEL[key]}: {entry['missing']}개 조각 미완료")
            out += [f"{LABEL[key]}: {p}" for p in entry["problems"]]
        return out

    system_check = state.get("system_check")
    system_ok = bool(system_check and system_check["ok"])
    if not system_ok:  # the request: leave the 시스템 row empty when it does not match the real system
        for config in ("system", *ML):
            for entry in results.get(config, {}).values():
                entry["problems"].append("시스템 동작 확인 실패" if system_check else "시스템 동작 확인을 하지 않음")
    not_run = {config: problems_of(config) for config, _ in CONFIGS + CHECKS if problems_of(config)}

    # ---- datasets.md
    rows = []
    totals = {"sessions": 0, "items": 0, "scored": 0, "unscored": 0, "types": Counter(), "channels": Counter()}
    for key in LABEL:
        if key not in stats:
            rows.append([LABEL[key]] + ["—"] * 7)
            continue
        s = stats[key]
        kinds = " + ".join(ORIGIN_KIND.get(o, str(o)) for o, _ in s["origins"].most_common())
        rows.append([LABEL[key], kinds, s["sessions"], s["items"], s["scored"], s["unscored"], dist(s["types"], TYPES),
                     dist(s["channels"], SCORED)])
        for k in ("sessions", "items", "scored", "unscored"):
            totals[k] += s[k]
        totals["types"].update(s["types"])
        totals["channels"].update(s["channels"])
    rows.append(["합계", "", totals["sessions"], totals["items"], totals["scored"], totals["unscored"],
                 dist(totals["types"], TYPES), dist(totals["channels"], SCORED)])
    missing_sets = [LABEL[k] for k in LABEL if k not in stats]
    doc = ["# 테스트셋 구성", "",
           table(["데이터셋", "출처 종류", "세션 수", "item 수", "채점 정답 수", "채점 안 하는 정답 수", "유형 분포", "채널 분포"], rows), "",
           "- 출처 종류는 세션의 `meta.origin`이다(authored = 직접 작성, external = 외부 변환, recorded = 실제 기록).",
           "- 채점 정답은 prompt·instructions·tool_output 채널의 정답이다. 채점 안 하는 정답은 tool_input(에이전트가 쓴 명령) 채널의 정답이다.",
           "- 유형 분포와 채널 분포는 채점 정답 기준이다.",
           "- privesc-llm-data의 평가용 변환본(`sessions_from_privesc-llm-data`)은 넣지 않았다. privesc는 ML 학습용 변환본 10개 시나리오 전체가 ML 평가 test_ood에 들어 있다(아래 주석)."]
    if missing_sets:
        doc.append(f"- {', '.join(missing_sets)}: 만들지 못했다. {state.get('dataset_notes', {}).get('missing', '노트북의 데이터 생성 단계 로그를 본다')}")
    for note in state.get("dataset_notes", {}).get("notes", []):
        doc.append(f"- {note}")
    doc += ["", "## 파일", "", table(["데이터셋", "파일", "sessions sha256", "gold sha256"],
                                     [[LABEL[k], f["file"], f["sessions_sha256"], f["gold_sha256"]] for k in keys for f in stats[k]["files"]])]
    (REPORT / "datasets.md").write_text("\n".join(doc) + "\n", encoding="utf-8")

    # ---- detection.md
    header = ["구성"] + [LABEL[k] for k in keys] + ["전체(micro)"]
    doc = ["# 탐지 성능", "",
           "판정은 정확 일치다. 비밀값의 글자만 정확히 가려야 exact이고, recall은 exact 비율이다. precision은 가린 곳 중 정답 안에 들어간 비율이다. "
           "1,000줄당 과잉 마스킹은 정답과 겹치지 않는 가림의 수다. 채점 채널은 prompt, instructions, tool_output이다.", ""]
    if not system_ok:
        reason = "; ".join(system_check["problems"]) if system_check else "확인 단계를 실행하지 않음"
        doc += [f"**시스템 행은 비웠다**: 채점 결과가 실제 시스템 동작과 같은지 확인하지 못했다({reason}).", ""]
    for metric, title, digits in (("recall", "recall", 3), ("precision", "precision", 3), ("f1", "F1", 3), ("f2", "F2", 3),
                                  ("fp_per_1k", "1,000줄당 과잉 마스킹", 2)):
        rows = []
        for config in DISPLAY:
            overall = combined(results, config, keys)
            rows.append([NAME[config]] + [cell(results, config, k, metric, n_of, digits) for k in keys]
                        + [fmt(overall[metric] if overall else None, digits)])
        doc += [f"## {title}", "", table(header, rows), "",
                "- 전체(micro)는 모든 데이터셋의 수를 합쳐 계산한다. 정답 수가 많은 CredData의 비중이 크다.",
                "- openhands-feedback, SWE-Gym은 비밀이 거의 없는 데이터라 recall보다 과잉 마스킹이 중요하다.",
                "- 정답이 30개 미만인 데이터셋은 칸에 정답 수를 같이 적었다(n=…).", ""]
    rows = []
    for config in DISPLAY:
        overall = combined(results, config, keys)
        rows.append([NAME[config]] + [fmt(overall[s] if overall else None) for s in ("exact", "over", "partial", "missed")])
    doc += ["## 판정 비율 (데이터셋 전체 합산)", "", table(["구성", "exact", "over", "partial", "missed"], rows), ""]
    for title, field, order in (("유형별", "type", TYPES), ("채널별", "channel", SCORED)):
        rows = []
        for value in order:
            row = [value]
            hit_all = tot_all = 0
            for key in keys:
                entry = results.get("system", {}).get(key)
                if not usable(entry):
                    row.append("—")
                    continue
                spans = [s for s in entry["spans"] if s[field] == value]
                hit = sum(s["status"] == "exact" for s in spans)
                hit_all += hit
                tot_all += len(spans)
                row.append("—" if not spans else (f"{hit / len(spans):.3f}" + (f" (n={len(spans)})" if len(spans) < 30 else "")))
            row.append("—" if not tot_all or not all(usable(results.get("system", {}).get(k)) for k in keys)
                       else f"{hit_all / tot_all:.3f} (n={tot_all})")
            rows.append(row)
        doc += [f"## 시스템의 {title} recall", "", table([{"type": "유형", "channel": "채널"}[field]] + [LABEL[k] for k in keys] + ["전체"], rows), "",
                "- —는 그 데이터셋에 해당 정답이 없거나 시스템 행을 비운 경우다.", ""]
    for config in ("system", *ML):
        blocked = {k: e["blocked"] for k, e in results.get(config, {}).items() if e["blocked"]}
        if blocked:
            doc += [f"- {NAME[config]}: 탐지기 오류(ML 모듈 timeout·오류 포함)로 Final Guard가 막은 것으로 채점한 item "
                    f"{sum(blocked.values())}개({', '.join(f'{LABEL[k]} {v}' for k, v in blocked.items())}). 그 item의 정답은 over로 채점된다.", ""]
    doc += ["- 시스템 + v9, 시스템 + Qwen은 시스템 기본 설정에 ML 모듈 하나를 플러그인 자체 스위치(`PG_DETECTOR_ENABLE`)로 켠 것이다. "
            "ML 모듈은 후보 전용으로 합집합에 더해진다. 실행마다 그 모듈이 실제로 돌았는지 확인했다.", ""]
    if not_run:
        doc += ["## 비운 칸", ""] + [f"- {NAME[c]}: {'; '.join(p[:5])}" for c, p in not_run.items()] + [""]
    (REPORT / "detection.md").write_text("\n".join(doc) + "\n", encoding="utf-8")

    # ---- miss_groups.md / .json
    by_config = {config: misses(results, config, texts) for config in ("union3", "credsweeper", "system")}
    group_ids = [g for g, _ in GROUPS]
    data = {"rule_changes": RULE_CHANGES, "configs": {}}
    doc = ["# 놓친 정답의 그룹", "",
           "exact가 아닌 채점 대상 정답을 아래 순서로 검사해 처음 걸리는 그룹에 넣었다. 앞 문맥은 같은 줄에서 값 바로 앞 100자다"
           "(E 그룹의 'PRIVATE KEY'는 줄과 상관없이 값 앞 60자에서 찾는다).", "",
           table(["순서", "그룹", "이름"], [[i + 1, g, n] for i, (g, n) in enumerate(GROUPS)]), ""]
    for config in ("union3", "credsweeper", "system"):
        rows = by_config[config]
        valid = [k for k in keys if usable(results.get(config, {}).get(k))]
        counts = Counter((r["group"], r["dataset"]) for r in rows)
        big_counts = Counter((big(r["group"]), r["dataset"]) for r in rows)
        data["configs"][config] = {"datasets": valid,
                                   "groups": {g: {k: counts[g, k] for k in valid} for g in group_ids},
                                   "big_groups": {b: {k: big_counts[b, k] for k in valid} for b, _ in BIG}}
        doc += [f"## {NAME[config]}에서 놓친 정답", ""]
        if not valid:
            doc += ["—", "", f"- 이 구성의 결과가 없다: {'; '.join(problems_of(config)[:3]) or '실행하지 않음'}", ""]
            continue
        head = ["그룹"] + [LABEL[k] for k in valid] + ["합계"]
        doc += ["### 큰 그룹", "", table(head, [[(b if b == n else f"{b} {n}")] + [big_counts[b, k] for k in valid] + [sum(big_counts[b, k] for k in valid)]
                                                for b, n in BIG]), "",
                "### 세부 그룹", "", table(head, [[(g if g == GROUP_NAME[g] else f"{g} {GROUP_NAME[g]}")] + [counts[g, k] for k in valid]
                                                  + [sum(counts[g, k] for k in valid)] for g in group_ids]), ""]
        skipped = [LABEL[k] for k in keys if k not in valid]
        if skipped:
            doc += [f"- 결과가 없어 뺀 데이터셋: {', '.join(skipped)}", ""]

    # union3 → system
    union_rows, system_rows = by_config["union3"], by_config["system"]
    if union_rows is not None and usable_all(results, "union3", keys) and usable_all(results, "system", keys):
        system_missed = {span_key(r) for r in system_rows}
        union_missed = {span_key(r) for r in union_rows}
        fixed = Counter(r["group"] for r in union_rows if span_key(r) not in system_missed)
        new = Counter(r["group"] for r in system_rows if span_key(r) not in union_missed)
        doc += ["## 기존 3개 합집합 → 시스템", "", "시스템(shell-aware를 더한 합집합)이 메운 정답과 새로 놓친 정답이다. 그룹은 합집합 기준(새로 놓친 것은 시스템 기준)이다.", "",
                table(["그룹", "합집합이 놓침", "시스템이 메움", "시스템이 새로 놓침", "시스템이 놓침"],
                      [[(g if g == GROUP_NAME[g] else f"{g} {GROUP_NAME[g]}"), sum(1 for r in union_rows if r["group"] == g), fixed[g], new[g],
                        sum(1 for r in system_rows if r["group"] == g)] for g in group_ids]), "", COLUMN_NOTE, ""]
        data["union3_to_system"] = {"fixed": dict(fixed), "newly_missed": dict(new)}
    else:
        doc += ["## 기존 3개 합집합 → 시스템", "", "—", "", "- 두 구성 중 결과가 없는 데이터셋이 있어 계산하지 않았다.", ""]

    # system → system + ML
    data["system_to_ml"] = {}
    for config in ML:
        if usable_all(results, "system", keys) and usable_all(results, config, keys):
            ml_rows = misses(results, config, texts)
            ml_missed = {span_key(r) for r in ml_rows}
            sys_missed = {span_key(r) for r in system_rows}
            fixed = Counter(r["group"] for r in system_rows if span_key(r) not in ml_missed)
            new = Counter(r["group"] for r in ml_rows if span_key(r) not in sys_missed)
            doc += [f"## 시스템 → {NAME[config]}", "", f"ML 모듈을 켜서 메운 정답과 새로 놓친 정답이다. 그룹은 시스템 기준(새로 놓친 것은 {NAME[config]} 기준)이다.", "",
                    table(["그룹", "시스템이 놓침", "메움", "새로 놓침", f"{NAME[config]} 놓침"],
                          [[glabel(g), sum(1 for r in system_rows if r["group"] == g), fixed[g], new[g],
                            sum(1 for r in ml_rows if r["group"] == g)] for g in group_ids]), "", COLUMN_NOTE, ""]
            data["system_to_ml"][config] = {"fixed": dict(fixed), "newly_missed": dict(new)}
        else:
            doc += [f"## 시스템 → {NAME[config]}", "", "—", "",
                    f"- 두 구성의 결과가 모두 있어야 계산한다: {'; '.join((problems_of('system') + problems_of(config))[:3])}", ""]

    # adapter-only exact
    adapters = {"credsweeper": "adapter_credsweeper", "gitleaks": "adapter_gitleaks", "promptguard-rules": "adapter_rules"}
    if all(usable_all(results, c, keys) for c in adapters.values()):
        exact = {name: {(s["stem"], s["session_id"], s["item_id"], s["start"], s["end"])
                        for e in results[c].values() for s in e["spans"] if s["status"] == "exact"}
                 for name, c in adapters.items()}
        stem_key = {c["stem"]: c["dataset"] for c in chunks}
        only = {name: Counter(stem_key[k[0]] for k in exact[name] - set().union(*(exact[o] for o in adapters if o != name)))
                for name in adapters}
        doc += ["## 그 어댑터만 정확히 가린 정답 수", "", "시스템 앙상블의 어댑터 3개를 하나씩 단독으로 돌렸을 때, 다른 두 어댑터는 exact로 가리지 못한 정답이다.", "",
                table(["어댑터"] + [LABEL[k] for k in keys] + ["합계"],
                      [[name] + [only[name][k] for k in keys] + [sum(only[name].values())] for name in adapters]), ""]
        data["adapter_only_exact"] = {name: dict(only[name]) for name in adapters}
    else:
        missing = [f"{NAME[c]}({'; '.join(problems_of(c)[:2]) or '결과 없음'})" for c in adapters.values() if not usable_all(results, c, keys)]
        doc += ["## 그 어댑터만 정확히 가린 정답 수", "", "—", "", f"- 어댑터 3개의 단독 결과가 모두 있어야 계산한다. 빠진 것: {', '.join(missing)}", ""]

    # C shape
    c_rows = [r for r in union_rows if r["group"] == "C"]
    if c_rows:
        values = [texts.items[r["stem"], r["session_id"], r["item_id"]][1][r["start"]:r["end"]] for r in c_rows]
        shape = {"길이 8 미만": sum(len(v) < 8 for v in values), "엔트로피 3.5 이하": sum(entropy(v) <= 3.5 for v in values),
                 "특수문자 있음": sum(any(not ch.isalnum() for ch in v) for v in values),
                 "숫자 없음": sum(not any(ch.isdigit() for ch in v) for v in values)}
        doc += ["## C 그룹 값의 모양 (기존 3개 합집합 기준)", "", f"C 그룹 {len(values)}개 중 (겹쳐 셀 수 있다):", "",
                table(["모양", "수"], [[k, v] for k, v in shape.items()]), "",
                "- 엔트로피는 값 안 글자 분포의 Shannon 엔트로피(글자당 비트)다. 특수문자는 영문자·숫자가 아닌 글자다.", ""]
        data["c_shape"] = {"total": len(values), **shape}

    # examples
    doc += ["## 그룹별 예시 (기존 3개 합집합 기준)", "", "값은 ⟦유형:길이자⟧로 바꿨다. 앞 50자와 뒤 15자를 보인다. 같은 창 안의 다른 정답도 가렸다.", ""]
    rng = random.Random(7)
    data["examples"] = {}
    for g in group_ids:
        pool = [r for r in union_rows if r["group"] == g]
        if not pool:
            continue
        picked = rng.sample(pool, min(3, len(pool)))
        shown = [masked(texts, r["stem"], r["session_id"], r["item_id"], r["start"], r["end"]) for r in picked]
        data["examples"][g] = [{"dataset": LABEL[r["dataset"]], "context": s} for r, s in zip(picked, shown)]
        doc += ["**" + (g if g == GROUP_NAME[g] else f"{g} {GROUP_NAME[g]}") + "**", ""] + [f"- {LABEL[r['dataset']]}: `{s.replace('`', 'ˋ')}`" for r, s in zip(picked, shown)] + [""]

    # per-dataset summary
    doc += ["## 데이터셋별 요약 (기존 3개 합집합 기준)", ""]
    data["dataset_summary"] = {}
    for key in keys:
        mine = Counter(r["group"] for r in union_rows if r["dataset"] == key)
        total = sum(mine.values())
        if not usable(results.get("union3", {}).get(key)):
            continue
        if not total:
            sentence = f"{LABEL[key]}: 기존 3개 합집합이 놓친 정답이 없다."
        else:
            top = mine.most_common(2)
            parts = [f"{glabel(g)} {n}개({n / total:.0%})" for g, n in top]
            sentence = f"{LABEL[key]}: 놓친 {total}개 중 " + "와 ".join(parts) + "가 가장 많다."
        data["dataset_summary"][key] = sentence
        doc.append(f"- {sentence}")
    doc.append("")

    # review
    sample = review_sample(union_rows, texts) if union_rows else []
    data["review"] = {"sample": sample, "verdicts": review or {}}
    doc += ["## 분류 검수", ""]
    if not sample:
        doc += ["—", "", "- 기존 3개 합집합 결과가 없어 검수하지 않았다.", ""]
    elif not review:
        doc += ["**검수 미완료.** 무작위 30개(시드 20261029)를 아래에 뽑아 두었다. 노트북의 검수 셀에 판정을 넣고 보고서를 다시 만든다.", ""]
    else:
        wrong = [s for s in sample if review.get(s["id"], ("?",))[0] == "N"]
        checked = [s for s in sample if review.get(s["id"], ("?",))[0] in ("Y", "N")]
        patterns = Counter(f"{s['group']} → {review[s['id']][1] or '?'}" for s in wrong)
        doc += [f"무작위 {len(sample)}개 중 {len(checked)}개를 직접 보고 판정했다. 틀린 분류 {len(wrong)}개.", ""]
        if patterns:
            doc += ["자주 틀린 패턴:", ""] + [f"- {p}: {n}개" for p, n in patterns.most_common()] + [""]
    doc += ["### 규칙 수정", ""] + ([f"- {c}" for c in RULE_CHANGES] or ["- 고치지 않았다."]) + [""]
    if sample:
        doc += ["### 검수 표본", "", table(["id", "데이터셋", "규칙 그룹", "판정", "맞음", "올바른 그룹", "메모", "문맥"],
                                         [[s["id"], s["dataset"], s["group"], s["status"],
                                           (review or {}).get(s["id"], ("", "", ""))[0],
                                           (review or {}).get(s["id"], ("", "", ""))[1],
                                           (review or {}).get(s["id"], ("", "", ""))[2],
                                           "`" + s["context"].replace("`", "ˋ").replace("|", "¦") + "`"] for s in sample]), ""]
    (REPORT / "miss_groups.md").write_text("\n".join(doc) + "\n", encoding="utf-8")
    (REPORT / "miss_groups.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

    # ---- slide_numbers.md
    columns = [("합성 3종", list(SYNTH)), ("CredData", ["creddata"]), ("Nemotron", ["nemotron"]),
               ("Nosey Parker", ["noseyparker"]), ("openhands", ["openhands"]), ("ML test", ["ml_test"]),
               ("ML test_ood", ["ml_test_ood"])]
    tools = [("CredSweeper (ML off)", "credsweeper"), ("CredSweeper (ML on)", "credsweeper_ml"), ("gitleaks", "gitleaks"),
             ("자체 규칙 (promptguard-rules)", "adapter_rules"), ("기존 3개 합침", "union3"),
             ("시스템 기본 설정 (shell-aware 포함 합집합)", "system"), ("시스템 + v9", "system_v9"), ("시스템 + Qwen", "system_qwen")]

    def col(config, members):
        members = [m for m in members if m in keys]
        value = combined(results, config, members) if members else None
        return fmt(value["recall"] if value else None)

    doc = ["# 발표 문서에 넣을 숫자", "", "## 도구별 recall", "",
           table(["도구"] + [c for c, _ in columns], [[label] + [col(config, m) for _, m in columns] for label, config in tools]),
           "", "- 합성 3종은 내장 코퍼스, 템플릿 예시, 에이전트 세션 v2를 합친 micro 값이다.", "- 자체 규칙은 시스템 앙상블에서 promptguard-rules 어댑터만 돌린 값이다."]
    for label, config in tools:
        if config in EXCLUDED:
            doc.append(f"- {label}: {EXCLUDED[config]}")
    empty = [c for c, m in columns if not any(k in keys for k in m)]
    if empty:
        doc.append(f"- {', '.join(empty)}: 데이터셋을 만들지 못해 비웠다(datasets.md).")
    doc += ["", "## 기존 3개를 합쳐도 놓친 정답의 그룹별 수", ""]
    group_columns = [("합성 3종", list(SYNTH)), ("CredData", ["creddata"]), ("Nemotron", ["nemotron"]),
                     ("Nosey Parker", ["noseyparker"]), ("ML test_ood", ["ml_test_ood"])]
    union_valid = {k for k in keys if usable(results.get("union3", {}).get(k))}
    big_counts = Counter((big(r["group"]), r["dataset"]) for r in union_rows)
    rows = []
    for b, name in BIG[:5]:
        row = [f"{b}. {name}"]
        for _, members in group_columns:
            members = [m for m in members if m in keys]
            row.append(sum(big_counts[b, m] for m in members) if members and all(m in union_valid for m in members) else "—")
        rows.append(row)
    doc += [table(["그룹"] + [c for c, _ in group_columns], rows), ""]
    other = sum(n for (b, _k), n in big_counts.items() if b == "기타")
    if other:
        doc.append(f"- 어느 규칙에도 걸리지 않은 기타 {other}개는 표에 없다(miss_groups.md).")
    doc += ["", "## 데이터셋별 정답 수", "",
            table(["데이터셋", "세션 수", "채점 정답 수"], [[LABEL[k], stats[k]["sessions"], stats[k]["scored"]] for k in keys]
                  + [["합계", sum(stats[k]["sessions"] for k in keys), sum(stats[k]["scored"] for k in keys)]]), ""]
    (REPORT / "slide_numbers.md").write_text("\n".join(doc) + "\n", encoding="utf-8")

    # ---- README.md
    singles = ["credsweeper", "credsweeper_ml", "gitleaks"]
    big_keys = [k for k in keys if n_of[k] >= 30]
    single_values = [(combined(results, c, [k]), c, k) for c in singles for k in big_keys]
    single_values = [(v["recall"], c, k) for v, c, k in single_values if v]
    union = combined(results, "union3", keys)
    system = combined(results, "system", keys)
    footnotes = []
    if single_values and all(usable_all(results, c, big_keys) for c in singles):
        low, high = min(single_values), max(single_values)
        s1 = f"기존 도구를 하나만 쓰면 recall이 데이터셋에 따라 {low[0]:.3f}~{high[0]:.3f}이다."
        footnotes.append(f"문장 1: CredSweeper(ML off·on), gitleaks 단독을 정답 30개 이상인 데이터셋({', '.join(LABEL[k] for k in big_keys)})에서 본 범위다. "
                         f"최소 {NAME[low[1]]}·{LABEL[low[2]]}, 최대 {NAME[high[1]]}·{LABEL[high[2]]}.")
    else:
        s1 = "기존 도구를 하나만 쓰면 recall이 데이터셋에 따라 —~—이다."
        missing = [f"{NAME[c]}({'; '.join(problems_of(c)[:1]) or '결과 없음'})" for c in singles if not usable_all(results, c, big_keys)]
        footnotes.append(f"문장 1: CredSweeper(ML off·on)와 gitleaks 단독 결과가 모두 있어야 채운다. 빠진 것: {', '.join(missing)}")
    if union:
        s2 = f"기존 도구 3개를 합쳐도 recall은 {union['recall']:.3f}이고, 그 대신 1,000줄당 과잉 마스킹이 {union['fp_per_1k']:.2f}번으로 늘어난다."
        singles_fp = [(combined(results, c, keys) or {}).get("fp_per_1k") for c in ("adapter_credsweeper", "adapter_gitleaks", "adapter_rules")]
        if all(v is not None for v in singles_fp):
            footnotes.append("문장 2: 기존 3개는 시스템 앙상블의 credsweeper·gitleaks·자체 규칙 어댑터이고, 하나라도 잡으면 가린다(전체 micro). "
                             f"어댑터 단독의 1,000줄당 과잉 마스킹은 {', '.join(f'{v:.2f}' for v in singles_fp)}(credsweeper, gitleaks, 자체 규칙)이다.")
    else:
        s2 = "기존 도구 3개를 합쳐도 recall은 —이고, 그 대신 1,000줄당 과잉 마스킹이 —번으로 늘어난다."
        footnotes.append("문장 2: 기존 3개 합집합 결과가 없어 채우지 않았다.")
    if system and union:
        diff = system["recall"] - union["recall"]
        s3 = (f"우리 시스템은 recall {system['recall']:.3f}, precision {system['precision']:.3f}이다(전체 micro). "
              f"기존 3개 합집합보다 recall이 {abs(diff) * 100:.1f}%p {'높다' if diff >= 0 else '낮다'}.")
    else:
        s3 = "우리 시스템은 recall —, precision —이다(전체 micro). 기존 3개 합집합보다 recall이 — 높다/낮다."
        footnotes.append("문장 3: 시스템 또는 기존 3개 합집합 결과가 없어 채우지 않았다.")
    if union_rows and data.get("union3_to_system"):
        groups_big = Counter(big(r["group"]) for r in union_rows)
        top, count = groups_big.most_common(1)[0]
        fixed_big = sum(n for g, n in data["union3_to_system"]["fixed"].items() if big(g) == top)
        s4 = (f"합쳐도 놓치는 정답 중 가장 큰 그룹은 {top} {dict(BIG)[top]}({count / len(union_rows):.0%})이고, "
              f"시스템이 그중 {fixed_big}개를 메웠다.")
        footnotes.append(f"문장 4: 기존 3개 합집합이 놓친 {len(union_rows)}개(exact가 아닌 채점 정답)를 큰 그룹으로 나눈 것이다.")
    else:
        s4 = "합쳐도 놓치는 정답 중 가장 큰 그룹은 —(—%)이고, 시스템이 그중 —개를 메웠다."
        footnotes.append("문장 4: 기존 3개 합집합과 시스템 결과가 모두 있어야 계산할 수 있다.")

    env = environment()
    checks = []
    for config in ("oracle", "identity"):
        value = combined(results, config, keys)
        if not value:
            checks.append(f"{config}: 결과 없음")
            continue
        ok = (value["recall"] == 1 and value["precision"] == 1) if config == "oracle" else value["recall"] == 0
        checks.append(f"{config}: recall {fmt(value['recall'])}, precision {fmt(value['precision'])} → {'정상' if ok else '비정상'}")
    described = (system_check or {}).get("described") or {}
    progress = json.loads((WORK / "progress.json").read_text()) if (WORK / "progress.json").is_file() else {}
    doc = ["# 중간발표 탐지 성능 요약", "", f"> {s1}", ">", f"> {s2}", ">", f"> {s3}", ">", f"> {s4}", ""]
    ml_lines = []
    for config in ML:
        value = combined(results, config, keys)
        if value and system:
            ml_lines.append(f"{NAME[config]}: recall {value['recall']:.3f}, precision {value['precision']:.3f}, "
                            f"1,000줄당 과잉 마스킹 {value['fp_per_1k']:.2f} (시스템 대비 recall {(value['recall'] - system['recall']) * 100:+.1f}%p)")
        else:
            ml_lines.append(f"{NAME[config]}: — ({'; '.join(problems_of(config)[:2]) or '결과 없음'})")
    doc += [f"- {f}" for f in footnotes] + ["", "## ML 모듈을 켠 시스템 (전체 micro)", ""] + [f"- {line}" for line in ml_lines]
    doc += ["", "## 측정 환경", "",
            f"- 기기: {env['machine']}",
            f"- 평가한 시스템 커밋: promptguard-claude-demoV0 {env['system_commit']}",
            f"- 데이터 리비전: promptguard-data {env['data_commit']}. 파일별 sha256은 datasets.md, 외부 원본 리비전은 아래.",
            *[f"  - {line}" for line in state.get("sources", [])],
            f"- CredSweeper {env['credsweeper']}, gitleaks {env['gitleaks']}, Python {env['python']}",
            f"- ML 모델: promptguard-model {env['model_commit'] or '—'}; torch {env['torch']}, GPU {env['gpu']}. "
            "v9는 detector.json대로 CPU, Qwen은 device auto.",
            "- 탐지기 설정(시스템): " + (f"candidate_policy {described.get('candidate_policy')}, consensus {described.get('consensus')}, "
                                    f"어댑터 {', '.join(a['name'] + ' ' + str(a.get('version')) for a in described.get('adapters', []))}, "
                                    f"후보 전용 {', '.join(described.get('candidate_only_adapters', [])) or '없음'}, ML 어댑터 꺼짐"
                                    if described else "—"),
            "- 탐지기 실패 처리: Runtime V1처럼 Final Guard가 요청을 막은 것으로 채점(`--on-detector-failure block`, 채점기 기본값).",
            f"- 시스템 동작 확인: {'통과' if system_ok else '실패 또는 미실행'}"
            + (f" (worker HTTP와 채점기 안 탐지 결과를 {system_check['texts_compared']}개 텍스트에서 비교, Mod 치환 계약 테스트: {system_check['contract_test']})"
               if system_check else ""),
            f"- 채점기 점검: {'; '.join(checks)}",
            "", "## 단계별 걸린 시간", "",
            table(["단계", "시간"], [[name, f"{v['seconds'] / 60:.1f}분" + ("" if v.get("ok", True) else " (실패)")]
                                    for name, v in state.get("stages", {}).items()]
                  + [[f"채점: {NAME.get(c, c)} (조각 합계)", f"{s / 60:.1f}분"] for c, s in progress.get("seconds", {}).items()]),
            "", "## 돌리지 못한 구성과 이유", ""]
    doc += [f"- {NAME[c]}: {'; '.join(p[:5])}" for c, p in not_run.items()] or ["- 없음"]
    for note in state.get("notes", []):
        doc.append(f"- {note}")
    doc += ["", "## 다른 결과 파일", "", "- datasets.md: 테스트셋 구성", "- detection.md: 구성 × 데이터셋 탐지 성능",
            "- miss_groups.md, miss_groups.json: 놓친 정답의 그룹", "- slide_numbers.md: 발표 문서에 넣을 숫자", ""]
    (REPORT / "README.md").write_text("\n".join(doc) + "\n", encoding="utf-8")
    leak_check(texts)
    print(f"report written: {REPORT}")
    return {"sentences": [s1, s2, s3, s4], "not_run": not_run, "review": data["review"]}


def usable_all(results, config, keys):
    return all(usable(results.get(config, {}).get(k)) for k in keys)


def leak_check(texts):
    """Refuse a report that contains a gold value that looks like a secret: 12+ characters, or 6+ with a digit or a
    symbol. Shorter or all-letter values (CredData labels words such as a literal password field value) collide
    with ordinary words in the masked contexts."""
    def risky(value):
        return len(value) >= 12 or (len(value) >= 6 and any(not ch.isalpha() for ch in value) and value.strip() == value)
    values = {value for key, spans in texts.gold.items() for span in spans
              for value in [texts.items[key][1][span["start"]:span["end"]]] if risky(value)}
    for path in REPORT.glob("*"):
        content = path.read_text(encoding="utf-8")
        leaked = sum(1 for value in values if value in content)
        if leaked:
            path.unlink()
            raise RuntimeError(f"{path.name} contained {leaked} gold value(s); the file was deleted")


# ---------------------------------------------------------------- background commands

def bg(name, cmd, cwd=None):
    """Run a shell command detached with its own log (WORK/logs/<name>.log) and exit-code file."""
    log = LOGS / f"{name}.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    for path in (log, Path(f"{log}.exit")):
        path.unlink(missing_ok=True)
    wrapped = f"( {cmd} ) > {log} 2>&1; echo $? > {log}.exit"
    subprocess.Popen(["bash", "-c", wrapped], cwd=cwd, env=clean_env(), start_new_session=True)
    print(f"[{name}] started; log: {log}")
    return name


def wait(name, every=60):
    """Poll a bg() command, printing its last log line every `every` seconds; raise on a nonzero exit."""
    log = LOGS / f"{name}.log"
    exit_file = Path(f"{log}.exit")
    started = time.time()
    while not exit_file.is_file():
        time.sleep(every)
        lines = log.read_text(errors="replace").strip().splitlines() if log.is_file() else []
        print(f"[{name}] {(time.time() - started) / 60:.0f}분: {lines[-1][:160] if lines else '(출력 없음)'}", flush=True)
    code = int(exit_file.read_text().strip() or 1)
    if code:
        tail = "\n".join(log.read_text(errors="replace").strip().splitlines()[-15:])
        raise RuntimeError(f"[{name}] exit {code}; log tail:\n{tail}")
    print(f"[{name}] done")
    return code


def note(text, key="notes"):
    """Record a line for README (key="notes") or datasets.md (key="dataset_notes")."""
    state = load_state()
    if key == "dataset_notes":
        state.setdefault("dataset_notes", {}).setdefault("notes", [])
        if text not in state["dataset_notes"]["notes"]:
            state["dataset_notes"]["notes"].append(text)
    else:
        state.setdefault("notes", [])
        if text not in state["notes"]:
            state["notes"].append(text)
    save_state(state)


def ml_full_copies(specs):
    """Evaluation copies of whole ML training builds, every split included: specs = [(row, name, sessions, gold)]
    with row "test" or "ood". The scorer refuses meta.split train/val, so each session gets split "test" (an
    evaluation split; allowed_use recomputed by the dataset policy) and keeps its own split in meta.ml_split.
    Verify the originals (ml_train_build.py verify, ml_train_privesc.py verify) before copying."""
    sys.path.insert(0, str(MD / "scripts"))
    from dataset_policy import allowed_use
    written = []
    for row, name, sessions, gold in specs:
        stem = f"mlfull_{row}_{name}"
        target = MD / "data_test" / f"sessions_{stem}.jsonl"
        target.parent.mkdir(parents=True, exist_ok=True)
        splits = Counter()
        with open(sessions, encoding="utf-8") as src, open(target, "w", encoding="utf-8") as out:
            for line in src:
                if not line.strip():
                    continue
                record = json.loads(line)
                meta = record.setdefault("meta", {})
                splits[meta.get("split")] += 1
                meta["ml_split"] = meta.get("split")
                meta["split"] = "test"
                meta["allowed_use"] = allowed_use(meta["origin"], "test")
                out.write(json.dumps(record, ensure_ascii=False) + "\n")
        (MD / "data_answer").mkdir(parents=True, exist_ok=True)
        shutil.copyfile(gold, MD / "data_answer" / f"gold_{stem}.jsonl")
        written.append(target)
        print(f"{target.name}: " + ", ".join(f"{k} {v}" for k, v in splits.items()))
    return written


def ml_smoke(configs=None):
    """Score the smallest chunk with each ML configuration before the full run: loads the model (Qwen downloads
    its base model the first time) and checks the module really votes. Raises with the system's reason if not."""
    configs = [c for c in (configs or list(ML)) if c not in EXCLUDED]
    chunk = min(make_chunks(), key=lambda c: c["weight"])
    script = WORK / "run_chunk.py"
    script.write_text(RUN_CHUNK.replace("__COMPOSE__", repr(COMPOSE)).replace("__ML__", repr(ML)))
    report = {}
    for config in configs:
        out = WORK / "smoke" / config
        shutil.rmtree(out, ignore_errors=True)
        started = time.time()
        completed = subprocess.run([sys.executable, str(script), config, chunk["sessions"], chunk["gold"], str(out),
                                    str(SYSTEM_ROOT), str(REPO), str(GITLEAKS_CACHE)],
                                   capture_output=True, text=True, env=clean_env())
        lines = (completed.stdout + completed.stderr).strip().splitlines()
        if completed.returncode:
            reason = next((line for line in lines if "error" in line.lower()), lines[-1] if lines else "")
            raise RuntimeError(f"{NAME[config]}: exit {completed.returncode}: {reason}")
        run = next(out.glob("*/run_meta.json")).parent
        system = json.loads((run / "run_meta.json").read_text())["system"]
        adapters = [a["name"] for a in system["detector"]["adapters"]]
        report[config] = {"adapters": adapters, "blocked": system.get("blocked_items", 0),
                          "seconds": round(time.time() - started, 1), "items": chunk["weight"]}
        print(f"{NAME[config]}: {adapters}, 차단 {report[config]['blocked']}, {report[config]['seconds']}초 ({chunk['stem']} {chunk['weight']} item)")
    state = load_state()
    state["ml_smoke"] = report
    save_state(state)
    return report

In [ ]:
import importlib, sys
sys.path.insert(0, "/content")
import pg_report as R
importlib.reload(R)
print("helper loaded:", R.REPO, R.SYSTEM_ROOT, R.WORK, R.REPORT)

## ① 설정

In [ ]:
import os

# 결과를 이어서 쓰고 보관할 Drive 폴더. 데이터 압축본, 채점 결과, 보고서가 여기에 남는다(비밀값이 든 데이터 포함. 공유하지 않는다)
DRIVE = "/content/drive/MyDrive/promptguard/report-midterm"

DATA_BRANCH = "refactor/eval-plugin-as-configured"  # promptguard-data. main에 머지한 뒤에는 "main"
SYSTEM_BRANCH = "main"                              # promptguard-claude-demoV0
MODEL_BRANCH = "main"                               # promptguard-model (v9, Qwen 가중치)

JOBS = os.cpu_count()  # 동시에 도는 채점 프로세스 수

# ML 평가셋의 LLM 템플릿(templates/ml_train/v1.jsonl). git에 없다. 받아 둔 파일이 있으면 Drive 경로를 넣는다
V1_TEMPLATES = None  # 예: "/content/drive/MyDrive/promptguard/v1.jsonl"

# 이미 따로 분석해서 이번에 돌리지 않는 구성. 보고서에서 그 칸은 —와 이 이유로 남는다
# (구성 이름: system, credsweeper, credsweeper_ml, gitleaks, adapter_credsweeper, adapter_gitleaks, adapter_rules, union3, majority3)
ALREADY_DONE = ["credsweeper", "gitleaks", "adapter_rules"]
EXCLUDED = {name: "이미 별도로 분석함, 이번 실행에서 제외" for name in ALREADY_DONE}
R.EXCLUDED = EXCLUDED

# 채점할 구성. None이면 제외한 것을 뺀 전부(우선순위: 시스템 → 기존 3개 합집합 → CredSweeper+ML → 기존 다수결 → 나머지 → oracle, identity)
CONFIGS = None

## ② 저장소와 Drive

세 저장소(데이터, 시스템, 모델)를 `/content` 아래에 나란히 받는다. 시스템의 ML 모듈은 `src/detector` 기준 `../../../promptguard-model`에서 모델을 찾는다. 외부 원본도 같은 곳(`/content/CredData` 등)에 받으므로, 변환 스크립트의 기본 경로가 그대로 맞는다.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, json
drive.mount("/content/drive")
DRIVE = Path(DRIVE); DRIVE.mkdir(parents=True, exist_ok=True)

R.sh("git lfs install -q || (apt-get -qq install -y git-lfs && git lfs install -q)")
for name, branch in (("promptguard-data", DATA_BRANCH), ("promptguard-claude-demoV0", SYSTEM_BRANCH),
                     ("promptguard-model", MODEL_BRANCH)):
    path = Path("/content") / name
    if not path.is_dir():
        R.sh(f"git clone -q -b {branch} https://github.com/capstone-privai/{name}.git {path}")
    else:
        R.sh(f"git -C {path} fetch -q origin {branch} && git -C {path} checkout -q FETCH_HEAD")
    R.sh(f"git -C {path} log -1 --format='{name}: %h %cs %s'")
R.sh("cd /content/promptguard-model && git lfs pull")
for weights in ("models/span-detector-v9/model.safetensors", "models/qwen-span-v2/adapter_model.safetensors"):
    size = (Path("/content/promptguard-model") / weights).stat().st_size
    print(f"{weights}: {size / 1e6:.1f} MB")
    assert size > 1_000_000, f"{weights}가 LFS 포인터 상태다(git lfs pull 확인)"

# 앞 세션의 단계별 시간·확인 결과를 이어받는다
R.WORK.mkdir(parents=True, exist_ok=True)
if (DRIVE / "state.json").is_file() and not R.STATE.is_file():
    shutil.copy(DRIVE / "state.json", R.STATE)

def save_to_drive():
    """state, 채점 진행, 보고서를 Drive에 복사한다."""
    for name in ("state.json", "progress.json"):
        if (R.WORK / name).is_file():
            shutil.copy(R.WORK / name, DRIVE / name)
    if R.REPORT.is_dir():
        shutil.copytree(R.REPORT, DRIVE / "report" / "detection", dirs_exist_ok=True)

## ③ 패키지

`credsweeper`는 시스템이 고정한 1.19.0이어야 한다. 설치 뒤 numpy 등이 바뀌었다는 경고가 나오면 **런타임 → 세션 다시 시작**을 하고 ⓪부터 다시 실행한다(받아 둔 것은 건너뛴다). gitleaks는 서버와 같은 8.30.1이다. node는 채점기의 Mod 치환 계약 테스트(⑥)에 쓴다.

In [ ]:
with R.stage("패키지 설치"):
    R.sh('pip -q install "credsweeper==1.19.0" "datasketch==2.0.0" "pybase62==1.0.0" "huggingface_hub[cli]>=0.34" pyarrow PyYAML '
         '"transformers>=4.51,<5" "peft>=0.13,<1"')  # torch는 Colab에 있다
    if R.sh("gitleaks version", check=False):
        R.sh("curl -sSL https://github.com/gitleaks/gitleaks/releases/download/v8.30.1/gitleaks_8.30.1_linux_x64.tar.gz"
             " | tar -xz -C /usr/local/bin gitleaks")
    R.sh("gitleaks version")
    R.sh("node --version", check=False)
    import importlib.metadata as md
    print("credsweeper", md.version("credsweeper"), "| torch", md.version("torch"), "| transformers", md.version("transformers"))
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 — Qwen이 CPU로 돌아 매우 느리다. GPU 런타임을 권장한다")

## ④ 외부 원본 받기

각 원본의 README(`metric-detection/README_<소스>.md`)에 고정된 리비전으로 받는다. 오래 걸리므로 백그라운드로 돌리고 1분마다 로그 마지막 줄을 보여 준다.
- CredData는 337개 저장소를 받는다(임시 폴더 약 13GB, 수십 분).
- Hugging Face에서 401/403이 나면 Colab 보안 비밀에 `HF_TOKEN`을 넣고, 그 데이터셋 페이지에서 이용 조건에 동의한 뒤 다시 실행한다.

Drive에 이 커밋으로 만든 데이터 압축본이 있으면 ④·⑤를 건너뛴다(⑤ 끝에서 저장한다).

In [ ]:
import subprocess
DATA_COMMIT = subprocess.run(f"git -C {R.REPO} rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
V1_TAG = "with-v1" if V1_TEMPLATES else "no-v1"
DATA_CACHE = DRIVE / f"built-data-{DATA_COMMIT}-{V1_TAG}.tar.gz"
DATA_READY = DATA_CACHE.is_file()
if DATA_READY:
    R.sh(f"rm -rf {R.MD}/data_test {R.MD}/data_answer && tar -xzf {DATA_CACHE} -C {R.MD}")
    print("이 커밋으로 만든 데이터를 Drive에서 복원했다. ④·⑤는 건너뛴다.")

SOURCES = [
    "CredData 0b1940e171725ad8937311120b191602608a4801",
    "Nemotron-PII b70ffaf5ff39e079776134c5bf4381f00a9fd1ed (data/test-*)",
    "openhands-feedback facf45600d625510b98222ec584a8c9ac59272a2",
    "SWE-Gym/OpenHands-SFT-Trajectories 4aaa5a4a4b5861f4799d2336908760c190ac3b17",
    "noseyparker 2e6e7f36ce36619852532bbe698d8cb7a26d2da7",
    "privesc-llm-data 39a9d2ff37222184fcaf2a6d9a124906e8bb49ad (ML 평가 test_ood의 두 시나리오만)",
]
state = R.load_state(); state["sources"] = SOURCES; R.save_state(state)

if not DATA_READY:
    with R.stage("원본 받기"):
        R.bg("download_hf", " && ".join([
            "hf download nvidia/Nemotron-PII --repo-type dataset --revision b70ffaf5ff39e079776134c5bf4381f00a9fd1ed "
            "--include 'data/test-*' --local-dir /content/Nemotron-PII",
            "hf download all-hands/openhands-feedback --repo-type dataset "
            "--revision facf45600d625510b98222ec584a8c9ac59272a2 --local-dir /content/openhands-feedback",
            "hf download SWE-Gym/OpenHands-SFT-Trajectories --repo-type dataset "
            "--revision 4aaa5a4a4b5861f4799d2336908760c190ac3b17 --local-dir /content/SWE-Gym-OpenHands-SFT-Trajectories",
            "hf download sailab-vienna/privesc-llm-data --repo-type dataset "
            "--revision 39a9d2ff37222184fcaf2a6d9a124906e8bb49ad --local-dir /content/privesc-llm-data",
        ]))
        R.bg("download_noseyparker", " && ".join([
            "rm -rf /content/noseyparker",
            "git clone -q --filter=blob:none --sparse https://github.com/praetorian-inc/noseyparker /content/noseyparker",
            "git -C /content/noseyparker checkout -q 2e6e7f36ce36619852532bbe698d8cb7a26d2da7",
            "git -C /content/noseyparker sparse-checkout set --no-cone '/crates/noseyparker/data/default/builtin/rules/*' '/LICENSE*'",
        ]))
        R.bg("download_creddata", " && ".join([
            "test -d /content/CredData/.git || git clone -q https://github.com/Samsung/CredData /content/CredData",
            "cd /content/CredData && git checkout -q 0b1940e171725ad8937311120b191602608a4801",
            "python download_data.py --jobs 8",
            "rm -rf /content/CredData/tmp",
        ]))
        R.wait("download_noseyparker", every=10)
        R.wait("download_hf")
        R.wait("download_creddata")

## ⑤ 데이터 생성과 검증

`metric-detection/README.md`의 "생성과 검증" 순서대로 만든다. 생성은 결정적이라 같은 커밋이면 누가 만들어도 바이트가 같다. privesc-llm-data 원본(`sessions_from_privesc-llm-data`)은 평가셋이 아니라 만들지 않는다.

ML 평가셋은 ML 학습 데이터 빌드 결과를 **split을 나누지 않고 전부** 쓴다. 평가하는 탐지기들은 이 데이터로 학습하지 않았다.
- **ML 평가 test:** 손 템플릿(hand_v1) 전체 800세션. LLM 템플릿(v1)이 있으면 v1 전체도 포함한다.
- **ML 평가 test_ood:** privesc 10개 시나리오 전체 2,200세션(ML 학습용 변환본, tool_input은 가림 표시).

채점기는 `meta.split`이 train·val인 세션을 거부한다. 그래서 원본을 검증한 뒤 평가용 사본(`sessions_mlfull_*`)에서는 split을 `test`로 두고, 원래 split은 `meta.ml_split`에 남긴다. v1은 Message Batches API로 쓴 결과물이라 git에 없다.

In [ ]:
SCRIPTS = "metric-detection/scripts"
VERIFY_ARGS = {
    "sessions_from_CredData": "--creddata /content/CredData",
    "sessions_from_Nemotron-PII": "--nemotron /content/Nemotron-PII",
    "sessions_from_openhands-feedback": "--openhands /content/openhands-feedback",
    "sessions_from_SWE-Gym": "--swe-gym /content/SWE-Gym-OpenHands-SFT-Trajectories",
    "sessions_from_noseyparker": "--noseyparker /content/noseyparker",
}
if not DATA_READY:
    with R.stage("데이터 생성"):
        R.sh(f"rm -rf {R.MD}/data_test {R.MD}/data_answer {R.MD}/ml_build {R.WORK}/ml_handoff")
        steps = [
            f"python {SCRIPTS}/dataset_build.py",
            f"python {SCRIPTS}/dataset_build.py --template metric-detection/templates/example.jsonl --origin authored",
            f"python {SCRIPTS}/dataset_build.py --template metric-detection/templates/agent_sessions_2.jsonl --origin authored --name 2",
            f"python {SCRIPTS}/convert_creddata.py --creddata /content/CredData",
            f"python {SCRIPTS}/convert_nemotron_pii.py --nemotron /content/Nemotron-PII",
            f"python {SCRIPTS}/convert_openhands_feedback.py --openhands /content/openhands-feedback",
            f"python {SCRIPTS}/convert_swe_gym.py --swe-gym /content/SWE-Gym-OpenHands-SFT-Trajectories",
            f"python {SCRIPTS}/convert_noseyparker.py --noseyparker /content/noseyparker",
        ]
        templates = ["metric-detection/templates/ml_train/hand_v1.jsonl"]
        if V1_TEMPLATES:
            shutil.copy(V1_TEMPLATES, R.REPO / "metric-detection/templates/ml_train/v1.jsonl")
            templates.insert(0, "metric-detection/templates/ml_train/v1.jsonl")
        steps.append(f"python {SCRIPTS}/ml_train_build.py build {' '.join(templates)} --out metric-detection/ml_build")
        # train 세션은 모두 넣는다(--sessions 큰 값). test_ood 440세션은 train 크기와 상관없이 전부 들어간다
        steps.append(f"python {SCRIPTS}/ml_train_privesc.py build --privesc /content/privesc-llm-data --out metric-detection/ml_build --sessions 100000")
        steps.append(f"python {SCRIPTS}/ml_train_build.py verify " + " ".join(
            f"metric-detection/ml_build/data_test/sessions_ml_{Path(t).stem}.jsonl" for t in templates))
        R.bg("build_data", " && ".join(steps), cwd=R.REPO)
        R.wait("build_data", every=30)
        if R.sh(f"python {SCRIPTS}/ml_train_privesc.py verify --privesc /content/privesc-llm-data --out metric-detection/ml_build",
                cwd=R.REPO, log=R.LOGS / "verify_ml_privesc.log", check=False):
            raise RuntimeError("ML privesc 빌드 검증 실패. 로그: " + str(R.LOGS / "verify_ml_privesc.log"))
        build = R.MD / "ml_build"
        R.ml_full_copies([("test", Path(t).stem, build / f"data_test/sessions_ml_{Path(t).stem}.jsonl",
                           build / f"data_answer/gold_ml_{Path(t).stem}.jsonl") for t in templates]
                         + [("ood", "privesc", build / "data_test/sessions_ml_privesc.jsonl", build / "data_answer/gold_ml_privesc.jsonl")])
    R.note("ML 평가 test는 ML 학습 데이터 빌드 전체(split 무시)다: 손 템플릿(hand_v1) 800세션" + (", LLM 템플릿(v1) 전체" if V1_TEMPLATES else "")
           + ". ML 평가 test_ood는 privesc-llm-data 10개 시나리오 전체 2,200세션(ML 학습용 변환본)이다. 평가하는 탐지기들은 이 데이터로 학습하지 않았다. "
           "채점기가 train·val split을 거부하므로 평가용 사본에서 split을 test로 바꾸고 원래 split은 meta.ml_split에 남겼다.", key="dataset_notes")
    if not V1_TEMPLATES:
        R.note("LLM 템플릿(v1)은 Message Batches API로 쓴 결과물이라 저장소에 없어 ML 평가 test에 넣지 못했다.", key="dataset_notes")

    with R.stage("데이터 검증"):
        failures = []
        for path in sorted((R.MD / "data_test").glob("sessions*.jsonl")):
            if path.name.startswith("sessions_mlfull_"):
                continue  # 원본은 ml_train_build.py verify, ml_train_privesc.py verify로 검증했다
            log = R.LOGS / f"verify_{path.stem}.log"
            log.unlink(missing_ok=True)
            code = R.sh(f"python {SCRIPTS}/dataset_verify.py {path} {VERIFY_ARGS.get(path.stem, '')}", cwd=R.REPO, log=log, check=False)
            print(f"  {path.name}: {'통과' if code == 0 else '실패'} (log {log})")
            if code:
                failures.append(path.name)
        if failures:
            raise RuntimeError(f"데이터 검증 실패: {failures}. 로그를 보고 원인을 확인한다. 추측으로 진행하지 않는다.")

    R.sh(f"tar -czf {DATA_CACHE} -C {R.MD} data_test data_answer")
    print("Drive에 저장:", DATA_CACHE)

found, stray = R.datasets()
for key, files in found.items():
    print(f"{R.LABEL[key]:22} {', '.join(stem for stem, *_ in files) or '— (없음)'}")
if stray:
    print("평가에서 빼는 파일:", stray)
save_to_drive()

## ⑥ 시스템 동작 확인

"시스템" 행이 실제 시스템 동작과 같은지 먼저 확인한다. 실패하면 보고서의 시스템 행을 비우고 이유를 적는다.
1. **설정:** 시스템 자체 설정이 합집합(`candidate_policy: union`)이고, 설정된 어댑터 4개(gitleaks 포함)가 모두 돌고, 빠진 어댑터가 없다.
2. **탐지 결과:** 실제 worker(`src/detector/worker.py`)를 HTTP로 띄워서 받은 탐지 결과가, 채점기가 프로세스 안에서 얻는 결과와 같다(데이터셋마다 무작위 40개 텍스트).
3. **치환 규칙:** 채점기의 Mod 치환 규칙이 실제 `register.mjs`와 같다(채점기의 계약 테스트를 node로 실행).

In [ ]:
with R.stage("시스템 동작 확인"):
    SYSTEM_CHECK = R.check_system(sample_per_dataset=40)
save_to_drive()
if not SYSTEM_CHECK["ok"]:
    print("시스템 행은 비워진다. 원인:", SYSTEM_CHECK["problems"])

ML 모듈 확인. 가장 작은 조각으로 "시스템 + v9"와 "시스템 + Qwen"을 한 번씩 돌려 모델이 실제로 투표하는지 본다. Qwen은 처음에 베이스 모델(약 1GB)을 받는다. 실패하면 플러그인이 알려 준 이유가 나온다. 예: 패키지 없음, 가중치가 LFS 포인터 상태. 고친 뒤 다시 실행한다.

In [ ]:
R.EXCLUDED = EXCLUDED
with R.stage("ML 모듈 확인"):
    ML_SMOKE = R.ml_smoke()
save_to_drive()

## ⑦ 채점 (백그라운드)

데이터셋을 세션 단위 조각(채점 item 약 1,500개)으로 나눠 구성 × 조각마다 채점기를 한 번씩 돌린다.
- **진행:** 진행 상황은 `/content/work/progress.json`, 조각별 로그는 `/content/work/logs/<구성>/`에 남는다.
- **이어서 하기:** 끝난 조각은 Drive(`runs/`)에 복사돼서, 끊긴 뒤 다시 실행하면 이어서 한다.
- **ML 구성:** "시스템 + v9"와 "시스템 + Qwen"은 같은 조각의 시스템 채점이 끝난 뒤 시작한다. Qwen은 GPU 메모리 때문에 한 번에 하나만 돈다. ML 모듈 timeout이나 오류가 난 item은 Final Guard가 막은 것으로 채점되고, 그 수가 보고서에 남는다.
- **gitleaks 캐시:** gitleaks 결과는 구성끼리 캐시로 공유한다. 대시보드의 `gitleaks_cache`를 쓰고, 같은 입력·같은 바이너리일 때만 재사용한다. 합집합·다수결·어댑터: gitleaks는 같은 조각의 시스템 채점이 끝난 뒤 시작한다.
- **확인:** 실행마다 실제로 돈 어댑터를 `run_meta.json`에서 확인한다. 예상과 다르면(예: gitleaks 빠짐) 그 칸은 쓰지 않고 —로 둔다.

In [ ]:
R.EXCLUDED = EXCLUDED  # ⓪을 다시 실행했어도 제외 목록을 유지한다
with R.stage("채점 준비"):
    CHUNKS = R.make_chunks()
    print(len(CHUNKS), "조각,", sum(c["weight"] for c in CHUNKS), "채점 item")
SCHEDULER_PID = R.start(configs=CONFIGS, jobs=JOBS, mirror=DRIVE / "runs", chunks=CHUNKS)

진행 확인. 끝날 때까지 10분마다 상태를 보여 준다(셀을 멈췄다 다시 실행해도 된다. 채점은 백그라운드에서 계속된다).

In [ ]:
import time
while True:
    progress = R.status()
    save_to_drive()
    if progress and progress["finished_at"]:
        break
    time.sleep(600)
if progress["failed"]:
    print("실패한 조각이 있다. 로그를 확인하고, 원인을 알기 전에는 다시 돌리지 않는다:", R.LOGS)

## ⑧ 보고서

In [ ]:
REVIEW = globals().get("REVIEW", {})
R.EXCLUDED = EXCLUDED
with R.stage("보고서 작성"):
    OUT = R.write_report(review=REVIEW)
save_to_drive()
for sentence in OUT["sentences"]:
    print(sentence)
print("\n보고서:", R.REPORT, "/ Drive:", DRIVE / "report" / "detection")

## ⑨ 분류 검수

기존 3개 합집합이 놓친 정답 중 무작위 30개(시드 20261029)를 보고, 규칙 그룹이 맞는지 판정한다. 문맥의 값은 가려져 있다. 판정은 아래 `REVIEW`에 넣는다.
- `"Y"`: 맞음
- `"N"`: 틀림. 올바른 그룹과 메모를 같이 적는다.

규칙을 고쳤다면 `R.RX`나 `R.classify`를 바꾸고, 무엇을 고쳤는지 `R.RULE_CHANGES`에 적는다. 그다음 ⑧을 다시 실행한다. 규칙을 바꾸면 표본도 다시 뽑히므로 그 표본으로 다시 검수한다.

In [ ]:
import pandas as pd
sample = json.loads((R.REPORT / "miss_groups.json").read_text())["review"]["sample"]
pd.set_option("display.max_colwidth", 200)
display(pd.DataFrame(sample)[["id", "dataset", "group", "status", "context"]])

In [ ]:
import re
# 판정: id → ("Y" 또는 "N", 올바른 그룹(N일 때, 예: "A-3"), 메모)
REVIEW = {
    # 1: ("Y", "", ""),
    # 2: ("N", "A-3", "함수 인자 자리인데 키 이름 규칙에 걸림"),
}
# 규칙을 고쳤다면 예:
# R.RX["shell"] = re.compile(r"...")
# R.RULE_CHANGES.append("A-2: echo 뒤 파이프가 없어도 셸 인자로 본다")

판정을 넣었으면 ⑧을 다시 실행한다.

## ⑩ 채팅 보고용 요약

In [ ]:
R.EXCLUDED = EXCLUDED
OUT = R.write_report(review=REVIEW)
save_to_drive()
print("■ README 맨 위 네 문장")
for sentence in OUT["sentences"]:
    print("-", sentence)
print("\n■ 돌리지 못한 것과 이유")
for config, problems in OUT["not_run"].items():
    print("-", R.NAME[config], ":", "; ".join(problems[:3]))
if not OUT["not_run"]:
    print("- 없음")
for line in R.load_state().get("notes", []) + R.load_state().get("dataset_notes", {}).get("notes", []):
    print("-", line)
review = OUT["review"]
done = [s for s in review["sample"] if review["verdicts"].get(s["id"], ("",))[0] in ("Y", "N")]
wrong = [s for s in done if review["verdicts"][s["id"]][0] == "N"]
print(f"\n■ 분류 검수: {len(done)}/{len(review['sample'])}개 판정, 틀린 분류 {len(wrong)}개" if done else "\n■ 분류 검수: 미완료")
print("\n■ 폴더:", R.REPORT, "(Drive:", DRIVE / "report" / "detection", ")")